## Overview: data preparation for individual-level regressions

Note: the resulting tables (commits, users, projects) are rather big. Therefore, all the preliminary preparation, cleaning and merging is done on the Cloud. Only clean and short task-specific tables are used in Stata to run regressions and to prepare summary statistics. 

* Regression preparations 
    * Outcomes: probability to contribute to a project by Russians, number of contributions to Russia, number of different projects with Russia, starting a new project (new owner) with Russia, quitting a project (owner) with Russia
    * Level: quarterly
    * Sample Ukrainian (Russian) users + control countries (diff-in-diff), control for time fixed effects, number of active authors, number of overall contributions, demand from other countries in terms of active projects and in terms of stars 
    * Heterogeneity: productivity of contributors and projects --> quality of c and pm previous work on given project, identity, name_identity, experience with russian/ukrainian owner generally (extensive), share of previous collab with ru/ua (intensive) --> outside options, users joining before 2014 --> dummy
   

In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
def check_costs(query):
    # settings
    dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
    run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

    # check processing costs
    job = client.query(query, job_config=dry)
    print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
    print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

In [ ]:
# individual quality and covariates
'''
# number of commits in the previous period (excluding self-commits)
# number of collaborations with this country  
# share of a country in contributions before 
# number of coauthors = number of project owners and coworkers on the same project 
# share of accepted PR in the previuos period 
# number of followers
# number of stars to the projects 
# experience on GH
# identity
# district 
# dlk
# name_class 
'''

### queries to create individual contributor- project owner country data set for regressions

In [ ]:
# comment H: level of individual user and owner, just owner id is dropped, no grouping yet

# previous commits by user and by country; 
# cumulative with SUM OVER (Partition ORDER BY)
query = '''
DROP VIEW if EXISTS `friends-foes.quality.user_commits`; 

CREATE VIEW `friends-foes.quality.user_commits` 
AS
SELECT DISTINCT year, month, author_id as user_id, country_author as country_user, country_owner as country_partner, commits_prev, commits_prev_country,
# number of previous owners to whom a commit was sent
SUM (first_owner) OVER(PARTITION BY author_id ORDER BY DATE(year, month, 01)) owners_prev, 
# number of previous owners from a given country to whom a commit was sent
SUM (first_owner) OVER(PARTITION BY author_id, country_owner ORDER BY DATE(year, month, 01)) owners_prev_country, 
FROM 
(
  SELECT year, month, author_id, country_author, owner_id, country_owner, commits,
  
  #one if first-time working with the owner
  CASE WHEN ROW_NUMBER() OVER (PARTITION BY author_id, owner_id ORDER BY DATE(year, month, 01)) = 1 THEN 1 ELSE 0
  END first_owner, 
  
  #number of previous commits
  SUM(commits) OVER(PARTITION BY author_id ORDER BY DATE(year, month, 01)) as commits_prev, 
  
  #number of previous commits to a given country
  SUM(commits) OVER(PARTITION BY author_id, country_owner ORDER BY DATE(year, month, 01)) commits_prev_country, 
  
  FROM `friends-foes.commits.puc` 
  WHERE owner_id <> author_id
)
ORDER BY author_id, country_owner, year, month
'''
check_costs(query)
#client.query(query, job_config=run) 

In [ ]:
# creates panel with cumulative number of followers of each user

# followers of a user
query = """
CREATE TABLE `friends-foes.quality.users_followers` 
AS
(
SELECT DISTINCT user_id, year, month, user_followers,
       SUM(user_followers) OVER(PARTITION BY user_id ORDER BY DATE(year, month, 01)) AS user_follow_cum
FROM
    (SELECT user_id, EXTRACT(YEAR FROM created_at) AS year,
           EXTRACT(MONTH FROM created_at) AS month, 
           COUNT(*) AS user_followers 
      FROM `ghtorrentmysql1906.MySQL1906.followers` 
    # delete followers form ru and ua  
    LEFT JOIN
        (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
        ON follower_id = u.id
    WHERE country_current NOT IN ('ru', 'ua') OR country_current IS NULL
    GROUP BY user_id, year, month
    )
)
"""
client.query(query, job_config=run) 

In [ ]:
# H: creates number of UA,RU,other followers of user pre 2014

# followers of a user by country-groups before 2014
query = """
CREATE TABLE `friends-foes.quality.users_followers_pre2014` 
AS
(
SELECT
  f.user_id,
  COUNT(*) AS total_followers_pre2014,
  COUNTIF(u.country_current = 'ru') AS followers_ru_pre2014,
  COUNTIF(u.country_current = 'ua') AS followers_ua_pre2014
FROM (
    SELECT *
    FROM `ghtorrentmysql1906.MySQL1906.followers`) f
LEFT JOIN (
  SELECT id, country_current
  FROM `friends-foes.user_panel.sample_locations_panel`
  WHERE year = 2019) u
ON f.follower_id = u.id # match to follower to get follower location
WHERE f.created_at < '2014-01-01'  -- all followers strictly before 2015
GROUP BY f.user_id
)

"""
check_costs(query)
client.query(query, job_config=run)

In [ ]:
# view for stars 

query = """
CREATE TABLE `friends-foes.quality.users_stars` 
AS
(
SELECT DISTINCT owner_id AS user_id, year, month, user_stars, 
       SUM(user_stars) OVER(PARTITION BY owner_id ORDER BY DATE(year, month, 01)) AS user_stars_cum
FROM 
  (
  SELECT p.owner_id, 
        EXTRACT(YEAR FROM created_at) AS year,
        EXTRACT(MONTH FROM created_at) AS month, 
        COUNT(*) AS user_stars  
    FROM `ghtorrentmysql1906.MySQL1906.watchers` s
    LEFT JOIN 
      (SELECT id, owner_id FROM `ghtorrentmysql1906.MySQL1906.projects` ) p
    ON s.repo_id = p.id
    LEFT JOIN 
      (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
    ON u.id = s.user_id 
    WHERE country_current NOT IN ('ru', 'ua') OR country_current IS NULL
    GROUP BY owner_id,year, month
  )
  WHERE owner_id IS NOT NULL
)
"""
client.query(query, job_config=run) 

In [ ]:
# creates number of UA,RU,other watchers/stars of user pre 2014


query = """
CREATE OR REPLACE TABLE `friends-foes.quality.users_stars_pre2014` AS
SELECT
  p.owner_id AS user_id,
  COUNT(DISTINCT s.user_id) AS total_stars_pre2014,
  COUNT(DISTINCT IF(u.country_current = 'ru', s.user_id, NULL)) AS stars_ru_pre2014,
  COUNT(DISTINCT IF(u.country_current = 'ua', s.user_id, NULL)) AS stars_ua_pre2014
FROM `ghtorrentmysql1906.MySQL1906.watchers` s
LEFT JOIN `ghtorrentmysql1906.MySQL1906.projects` p
  ON s.repo_id = p.id
LEFT JOIN (
  SELECT id, country_current
  FROM `friends-foes.user_panel.sample_locations_panel`
  WHERE year = 2019
) u
  ON s.user_id = u.id
WHERE s.created_at < '2014-01-01'
  AND p.owner_id IS NOT NULL
GROUP BY p.owner_id
"""
check_costs(query)
client.query(query, job_config=run)

In [ ]:
# contributors of the owner/user // will be joined to user_quality table
query = '''
CREATE VIEW `friends-foes.quality.user_projects` 
AS
SELECT DISTINCT year, month, owner_id AS user_id, country_owner as country_user, country_author AS country_partner, 
       contributions_prev, contributions_prev_country,

# number of previous users who sent a commit to an owner 
SUM (first_author) OVER(PARTITION BY owner_id ORDER BY DATE(year, month, 01)) as contributors_prev,

# number of previous owners from a given country to whom a commit was sent
SUM (first_author) OVER(PARTITION BY owner_id, country_author ORDER BY DATE(year, month, 01)) contributors_prev_country, 

#number of previous projects
  SUM(first_project) OVER(PARTITION BY owner_id ORDER BY DATE(year, month, 01)) as projects_prev, 

FROM 
(
  SELECT year, month, author_id, owner_id, country_author, country_owner,
  #number of previous contributions
  SUM(commits) OVER(PARTITION BY owner_id ORDER BY DATE(year, month, 01)) as contributions_prev, 
  #number of previous contributions from a given country
  SUM(commits) OVER(PARTITION BY owner_id, country_author ORDER BY DATE(year, month, 01)) contributions_prev_country,
  #one if first-time working with the author
  CASE WHEN ROW_NUMBER() OVER (PARTITION BY author_id, owner_id ORDER BY DATE(year, month, 01)) = 1 THEN 1 ELSE 0
  END first_author, 
  #one if first-time mention of an owner's project
  CASE WHEN ROW_NUMBER() OVER (PARTITION BY owner_id, parent_id ORDER BY DATE(year, month, 01)) = 1 THEN 1 ELSE 0
  END first_project
  FROM `friends-foes.commits.puc` WHERE author_id <> owner_id
)
ORDER BY user_id, country_partner, year, month
'''
client.query(query, job_config=run) 

In [ ]:
# create number of contributors to user from ru,ua,all before 2014

query = """
CREATE OR REPLACE TABLE `friends-foes.quality.users_contributors_pre2014` AS
SELECT owner_id AS user_id,
    COUNT (DISTINCT author_id) AS contributors,
    COUNT(DISTINCT IF(country_author = 'ua', author_id, NULL)) AS ua_contributors,
    COUNT(DISTINCT IF(country_author = 'ru', author_id, NULL)) AS ru_contributors   
FROM `friends-foes.commits.puc` 
WHERE author_id <> owner_id AND year < 2014
GROUP BY owner_id

"""
check_costs(query)
client.query(query, job_config=run)

In [ ]:
# select contributions for heterogeneity regressions; at author-id, country-partner, year, and company
# heterogeneity: location, name classification, joined after 2013
# controls: experience on GH (years: year - min(year)), num of prev. commits,  num of prev. country partners, 
# num of prev. owners

# for those who joined before 2014: take prev activity measures as of 2013; add previous work with Russians

# ukrainians

query = '''
DROP TABLE if EXISTS `friends-foes.commits.ua_indiv_analysis`; 

CREATE TABLE `friends-foes.commits.ua_indiv_analysis`
AS 
SELECT c.year, c.author_id, c.country_author, c.country_owner,
       CASE WHEN compb.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
       CASE WHEN compa.company_current IS NOT NULL THEN 1 ELSE 0 END company_owner,
       CASE WHEN c.country_owner in ('ua', 'us', 'ru', 'de', 'gb',  'ca', 'fr', 'cn', 'in') 
               THEN c.country_owner ELSE "other" END country_partner,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, 
       COUNT(DISTINCT c.owner_id) as n_owners, 
       COUNT(DISTINCT parent_id) AS n_projects
FROM `friends-foes.commits.puc` c

#company affiliation for author and owner
LEFT JOIN `friends-foes.user_panel.sample_companies` compa ON compa.year = c.year AND compa.id = c.owner_id
LEFT JOIN `friends-foes.user_panel.sample_companies` compb ON compb.year = c.year AND compb.id = c.author_id

WHERE country_author = 'ua' AND c.author_id <> c.owner_id and c.year between 2011 and 2018

GROUP BY c.year, c.author_id, country_author, c.country_owner, company_author, company_owner
'''
check_costs(query)
#client.query(query, job_config=run)

In [ ]:
# russians 
query = '''
DROP TABLE if EXISTS `friends-foes.commits.ru_indiv_analysis`; 

CREATE TABLE `friends-foes.commits.ru_indiv_analysis`
AS 
SELECT c.year, c.author_id, c.country_author, c.country_owner,  
       CASE WHEN compb.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
       CASE WHEN compa.company_current IS NOT NULL THEN 1 ELSE 0 END company_owner,
       CASE WHEN c.country_owner in ('us', 'ru', 'de', 'gb',  'ca', 'fr', 'cn', 'in') 
              THEN c.country_owner 
            WHEN c.country_owner = 'ua' AND ru_identity_owner = 1 
               THEN 'ua_east'
           WHEN c.country_owner = 'ua' AND ru_identity_owner > 0
                THEN 'ua_central'
            WHEN c.country_owner = 'ua' AND ru_identity_owner = 0
                THEN 'ua_west'
            ELSE "other" END country_partner,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, 
       COUNT(DISTINCT c.owner_id) as n_owners, 
       COUNT(DISTINCT parent_id) AS n_projects
FROM `friends-foes.commits.puc` c

#company affiliation for author and owner
LEFT JOIN `friends-foes.user_panel.sample_companies` compa ON compa.year = c.year AND compa.id = c.owner_id
LEFT JOIN `friends-foes.user_panel.sample_companies` compb ON compb.year = c.year AND compb.id = c.author_id

WHERE country_author = 'ru' AND c.author_id <> c.owner_id and c.year between 2011 and 2018 
    #  AND (dlk_owner = 0 OR dlk_owner is null)

GROUP BY c.year, c.author_id, country_author, c.country_owner, company_author, company_owner, ru_identity_owner

'''

check_costs(query)
#client.query(query, job_config=run)

In [ ]:
# make a balanced panel, year-level
# ukrainians
query = '''

CREATE OR REPLACE TABLE `friends-foes.commits.ua_indiv_analysis_balanced`
AS 


SELECT * FROM 
(
SELECT  balance.*, com.company_author, com.commits_int, com.commits_ext, com.n_owners, com.n_projects, 
       up.year_created_a, 
       up.cc2013_author, ua.russians, ua.language_ru, ua.district_en, ua.ru_identity, ua.dlk, 
       names.ethnicities, names.confidence, ma.mover_author, ma.stayer_author, 
       prev.commits_cum, prev.owners_cum, prev.countries_cum, prev_stars.stars_cum,
       networkf.total_followers_pre2014 ,networkf.followers_ru_pre2014 ,networkf.followers_ua_pre2014 ,
       networks.total_stars_pre2014 ,networks.stars_ru_pre2014 ,networks.stars_ua_pre2014 ,
       networkc.contributors ,networkc.ua_contributors ,networkc.ru_contributors 
       
FROM

# creates a balanced table
(SELECT a.author_id, b.country_owner, c.year, d.company_owner, e.min_year 
 FROM (SELECT DISTINCT author_id  FROM `friends-foes.commits.ua_indiv_analysis`) a 
 CROSS JOIN (SELECT * FROM UNNEST(['ua', 'us', 'ru', 'de', 'gb',  'ca', 'fr', 'cn', 'in', 'other']) AS country_owner) b
 CROSS JOIN (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) c 
 CROSS JOIN (SELECT * FROM UNNEST([0, 1]) AS company_owner) d
 LEFT JOIN 
     (SELECT author_id, min(year) as min_year FROM  `friends-foes.commits.ua_indiv_analysis`
      GROUP BY author_id) e
ON a.author_id = e.author_id
 ) balance

# add data with commits
LEFT JOIN 
       (SELECT author_id, year, country_author, company_author, company_owner, country_partner, 
       SUM(commits_int) as commits_int, SUM(commits_ext) as commits_ext, SUM(n_owners) as n_owners, SUM(n_projects) as n_projects
       FROM `friends-foes.commits.ua_indiv_analysis` 
       GROUP BY author_id, year, country_author, company_author, company_owner, country_partner
       ORDER BY author_id, year, country_partner) com 
ON com.author_id = balance.author_id and com.year = balance.year 
    and com.country_partner = balance.country_owner and com.company_owner = balance.company_owner
    
# add 2013 locations for users who were already on the platform back then
LEFT JOIN (SELECT id, min(year_created) AS year_created_a,
            min(CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #because cc2013 is not geocoded for all observations, it is replaced with the first non-missing countr
            END ) as cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            GROUP BY id)  up
ON balance.author_id = up.id

# additional info for UA
LEFT JOIN (SELECT id, year, district_en, ru_identity, dlk, russians, language_ru FROM `friends-foes.user_panel.ua_users_new`) ua
ON balance.author_id = ua.id and balance.year = ua.year

LEFT JOIN `friends-foes.user_panel.name_class_web` names
ON balance.author_id = names.id

# mover or stayer
LEFT JOIN (SELECT author_id, stayer AS stayer_author, mover AS mover_author FROM `friends-foes.commits.movers`) ma
ON balance.author_id = ma.author_id

#activity controls: cumulated commits, country partners, and owners 
LEFT JOIN (SELECT year, user_id, MAX(commits_prev) as commits_cum, MAX(owners_prev) as owners_cum, 
          COUNT(DISTINCT country_partner) as countries_cum FROM `friends-foes.quality.user_commits`
          GROUP BY year, user_id) prev
ON balance.author_id = prev.user_id and balance.year = prev.year

#quality controls: number of stars 
LEFT JOIN (SELECT year, user_id, MAX(user_stars_cum) as stars_cum FROM `friends-foes.quality.users_stars`
            GROUP BY year, user_id) prev_stars
ON balance.author_id = prev_stars.user_id and balance.year = prev_stars.year 

# below added by Helene
# network heterogeneity: followers
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_followers_pre2014`) networkf
ON balance.author_id = networkf.user_id

# network heterogeneity: stars
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_stars_pre2014`) networks
ON balance.author_id = networks.user_id

# network heterogeneity: contributors
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_contributors_pre2014`) networkc
ON balance.author_id = networkc.user_id
)

where year >= year_created_a

'''

check_costs(query)
#client.query(query, job_config=run)

In [ ]:

query = '''
SELECT * FROM `friends-foes.commits.ua_indiv_analysis_balanced`
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regression_p2c_ua_310523.csv', encoding = 'utf-8', index=False)


### below creates same data set for Ukraine but includes full set of control countries

In [ ]:
# select contributions for heterogeneity regressions; at author-id, country-partner, year, and company
# heterogeneity: location, name classification, joined after 2013
# controls: experience on GH (years: year - min(year)), num of prev. commits,  num of prev. country partners, 
# num of prev. owners

# for those who joined before 2014: take prev activity measures as of 2013; add previous work with Russians

# ukrainians

query = '''
DROP TABLE if EXISTS `friends-foes.commits.ua_indiv_analysis_fullcontrol`; 

CREATE TABLE `friends-foes.commits.ua_indiv_analysis_fullcontrol`
AS 
SELECT c.year, c.author_id, c.country_author, c.country_owner,
       CASE WHEN compb.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
       CASE WHEN compa.company_current IS NOT NULL THEN 1 ELSE 0 END company_owner,
       CASE WHEN c.country_owner in ('ua', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee', 'pl') 
               THEN c.country_owner ELSE "other" END country_partner,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, 
       COUNT(DISTINCT c.owner_id) as n_owners, 
       COUNT(DISTINCT parent_id) AS n_projects
FROM `friends-foes.commits.puc` c

#company affiliation for author and owner
LEFT JOIN `friends-foes.user_panel.sample_companies` compa ON compa.year = c.year AND compa.id = c.owner_id
LEFT JOIN `friends-foes.user_panel.sample_companies` compb ON compb.year = c.year AND compb.id = c.author_id

WHERE country_author = 'ua' AND c.author_id <> c.owner_id and c.year between 2011 and 2018

GROUP BY c.year, c.author_id, country_author, c.country_owner, company_author, company_owner
'''
check_costs(query)
client.query(query, job_config=run)

In [ ]:
# make a balanced panel, year-level, full set of control countries
# ukrainians
query = '''

CREATE OR REPLACE TABLE `friends-foes.commits.ua_indiv_analysis_balanced_fullcontrol`
AS 


SELECT * FROM 
(
SELECT  balance.*, com.company_author, com.commits_int, com.commits_ext, com.n_owners, com.n_projects, 
       up.year_created_a, 
       up.cc2013_author, ua.russians, ua.language_ru, ua.district_en, ua.ru_identity, ua.dlk, 
       names.ethnicities, names.confidence, ma.mover_author, ma.stayer_author, 
       prev.commits_cum, prev.owners_cum, prev.countries_cum, prev_stars.stars_cum,
       networkf.total_followers_pre2014 ,networkf.followers_ru_pre2014 ,networkf.followers_ua_pre2014 ,
       networks.total_stars_pre2014 ,networks.stars_ru_pre2014 ,networks.stars_ua_pre2014 ,
       networkc.contributors ,networkc.ua_contributors ,networkc.ru_contributors 
       
FROM

# creates a balanced table
(SELECT a.author_id, b.country_owner, c.year, d.company_owner, e.min_year 
 FROM (SELECT DISTINCT author_id  FROM `friends-foes.commits.ua_indiv_analysis_fullcontrol`) a 
 CROSS JOIN (SELECT * FROM UNNEST(['ua', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee', 'pl', 'other']) AS country_owner) b
 CROSS JOIN (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) c 
 CROSS JOIN (SELECT * FROM UNNEST([0, 1]) AS company_owner) d
 LEFT JOIN 
     (SELECT author_id, min(year) as min_year FROM  `friends-foes.commits.ua_indiv_analysis_fullcontrol`
      GROUP BY author_id) e
ON a.author_id = e.author_id
 ) balance

# add data with commits
LEFT JOIN 
       (SELECT author_id, year, country_author, company_author, company_owner, country_partner, 
       SUM(commits_int) as commits_int, SUM(commits_ext) as commits_ext, SUM(n_owners) as n_owners, SUM(n_projects) as n_projects
       FROM `friends-foes.commits.ua_indiv_analysis_fullcontrol` 
       GROUP BY author_id, year, country_author, company_author, company_owner, country_partner
       ORDER BY author_id, year, country_partner) com 
ON com.author_id = balance.author_id and com.year = balance.year 
    and com.country_partner = balance.country_owner and com.company_owner = balance.company_owner
    
# add 2013 locations for users who were already on the platform back then
LEFT JOIN (SELECT id, min(year_created) AS year_created_a,
            min(CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #because cc2013 is not geocoded for all observations, it is replaced with the first non-missing countr
            END ) as cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            GROUP BY id)  up
ON balance.author_id = up.id

# additional info for UA
LEFT JOIN (SELECT id, year, district_en, ru_identity, dlk, russians, language_ru FROM `friends-foes.user_panel.ua_users_new`) ua
ON balance.author_id = ua.id and balance.year = ua.year

LEFT JOIN `friends-foes.user_panel.name_class_web` names
ON balance.author_id = names.id

# mover or stayer
LEFT JOIN (SELECT author_id, stayer AS stayer_author, mover AS mover_author FROM `friends-foes.commits.movers`) ma
ON balance.author_id = ma.author_id

#activity controls: cumulated commits, country partners, and owners 
LEFT JOIN (SELECT year, user_id, MAX(commits_prev) as commits_cum, MAX(owners_prev) as owners_cum, 
          COUNT(DISTINCT country_partner) as countries_cum FROM `friends-foes.quality.user_commits`
          GROUP BY year, user_id) prev
ON balance.author_id = prev.user_id and balance.year = prev.year

#quality controls: number of stars 
LEFT JOIN (SELECT year, user_id, MAX(user_stars_cum) as stars_cum FROM `friends-foes.quality.users_stars`
            GROUP BY year, user_id) prev_stars
ON balance.author_id = prev_stars.user_id and balance.year = prev_stars.year 

# below added by Helene
# network heterogeneity: followers
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_followers_pre2014`) networkf
ON balance.author_id = networkf.user_id

# network heterogeneity: stars
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_stars_pre2014`) networks
ON balance.author_id = networks.user_id

# network heterogeneity: contributors
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_contributors_pre2014`) networkc
ON balance.author_id = networkc.user_id
)

where year >= year_created_a

'''

check_costs(query)
client.query(query, job_config=run)

In [ ]:

query = '''
SELECT * FROM `friends-foes.commits.ua_indiv_analysis_balanced_fullcontrol`
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regression_p2c_ua_251125.csv', encoding = 'utf-8', index=False)


### data set for Russia, subset of control countries

In [ ]:
# make a balanced panel
# russians
query = ''' 

CREATE OR REPLACE TABLE `friends-foes.commits.ru_indiv_analysis_balanced`
AS 


SELECT * FROM 
(
SELECT  balance.*, com.company_author, com.commits_int, com.commits_ext, com.n_owners, com.n_projects, 
       up.year_created_a, up.cc2013_author, ma.mover_author, ma.stayer_author, 
       prev.commits_cum, prev.owners_cum, prev.countries_cum, prev_stars.stars_cum,
       networkf.total_followers_pre2014 ,networkf.followers_ru_pre2014 ,networkf.followers_ua_pre2014 ,
       networks.total_stars_pre2014 ,networks.stars_ru_pre2014 ,networks.stars_ua_pre2014 ,
       networkc.contributors ,networkc.ua_contributors ,networkc.ru_contributors
FROM

# creates a balanced table
(SELECT a.author_id, b.country_owner, c.year, d.company_owner, e.min_year 
 FROM (SELECT DISTINCT author_id  FROM `friends-foes.commits.ru_indiv_analysis`) a 
 CROSS JOIN (SELECT * FROM UNNEST(['ua_west', 'ua_east', 'ua_central', 'us', 'ru', 'de', 'gb',  'ca', 'fr', 'cn', 'in', 'other']) AS country_owner) b
 CROSS JOIN (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) c 
 CROSS JOIN (SELECT * FROM UNNEST([0, 1]) AS company_owner) d
 LEFT JOIN 
     (SELECT author_id, min(year) as min_year FROM  `friends-foes.commits.ru_indiv_analysis`
      GROUP BY author_id) e
ON a.author_id = e.author_id
 ) balance

# add data with commits
LEFT JOIN 
       (SELECT author_id, year, country_author, company_author, company_owner, country_partner, 
       SUM(commits_int) as commits_int, SUM(commits_ext) as commits_ext, SUM(n_owners) as n_owners, SUM(n_projects) as n_projects
       FROM `friends-foes.commits.ru_indiv_analysis` 
       GROUP BY author_id, year, country_author, company_author, company_owner, country_partner
       ORDER BY author_id, year, country_partner) com 
ON com.author_id = balance.author_id and com.year = balance.year 
    and com.country_partner = balance.country_owner and com.company_owner = balance.company_owner
    
# add 2013 locations for users who were already on the platform back then
LEFT JOIN (SELECT id, min(year_created) AS year_created_a,
            min(CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #because cc2013 is not geocoded for all observations, it is replaced with the first non-missing countr
            END ) as cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            GROUP BY id)  up
ON balance.author_id = up.id


# mover or stayer
LEFT JOIN (SELECT author_id, stayer AS stayer_author, mover AS mover_author FROM `friends-foes.commits.movers`) ma
ON balance.author_id = ma.author_id

#activity controls: cumulated commits, country partners, and owners 
LEFT JOIN (SELECT year, user_id, MAX(commits_prev) as commits_cum, MAX(owners_prev) as owners_cum, 
          COUNT(DISTINCT country_partner) as countries_cum FROM `friends-foes.quality.user_commits`
          GROUP BY year, user_id) prev
ON balance.author_id = prev.user_id and balance.year = prev.year

#quality controls: number of stars 
LEFT JOIN (SELECT year, user_id, MAX(user_stars_cum) as stars_cum FROM `friends-foes.quality.users_stars`
            GROUP BY year, user_id) prev_stars
ON balance.author_id = prev_stars.user_id and balance.year = prev_stars.year 

# below added by Helene
# network heterogeneity: followers
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_followers_pre2014`) networkf
ON balance.author_id = networkf.user_id

# network heterogeneity: stars
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_stars_pre2014`) networks
ON balance.author_id = networks.user_id

# network heterogeneity: contributors
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_contributors_pre2014`) networkc
ON balance.author_id = networkc.user_id
)


where year >= year_created_a

'''
client.query(query, job_config=run) 


In [ ]:

query = '''
SELECT * FROM `friends-foes.commits.ru_indiv_analysis_balanced`
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regression_p2c_ru_050623-dlk.csv', encoding = 'utf-8', index=False)


### below creates same data set for Russia but includes full set of control countries

In [ ]:
# russians 
query = '''
DROP TABLE if EXISTS `friends-foes.commits.ru_indiv_analysis_fullcontrol`; 

CREATE TABLE `friends-foes.commits.ru_indiv_analysis_fullcontrol`
AS 
SELECT c.year, c.author_id, c.country_author, c.country_owner,  
       CASE WHEN compb.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
       CASE WHEN compa.company_current IS NOT NULL THEN 1 ELSE 0 END company_owner,
       CASE WHEN c.country_owner in ('ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee', 'pl') 
              THEN c.country_owner 
            WHEN c.country_owner = 'ua' AND ru_identity_owner = 1 
               THEN 'ua_east'
           WHEN c.country_owner = 'ua' AND ru_identity_owner > 0
                THEN 'ua_central'
            WHEN c.country_owner = 'ua' AND ru_identity_owner = 0
                THEN 'ua_west'
            ELSE "other" END country_partner,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, 
       COUNT(DISTINCT c.owner_id) as n_owners, 
       COUNT(DISTINCT parent_id) AS n_projects
FROM `friends-foes.commits.puc` c

#company affiliation for author and owner
LEFT JOIN `friends-foes.user_panel.sample_companies` compa ON compa.year = c.year AND compa.id = c.owner_id
LEFT JOIN `friends-foes.user_panel.sample_companies` compb ON compb.year = c.year AND compb.id = c.author_id

WHERE country_author = 'ru' AND c.author_id <> c.owner_id and c.year between 2011 and 2018 
    #  AND (dlk_owner = 0 OR dlk_owner is null)

GROUP BY c.year, c.author_id, country_author, c.country_owner, company_author, company_owner, ru_identity_owner

'''

check_costs(query)
client.query(query, job_config=run)

In [ ]:
# make a balanced panel, full set of control countries
# russians
query = ''' 

CREATE OR REPLACE TABLE `friends-foes.commits.ru_indiv_analysis_balanced_fullcontrol`
AS 


SELECT * FROM 
(
SELECT  balance.*, com.company_author, com.commits_int, com.commits_ext, com.n_owners, com.n_projects, 
       up.year_created_a, up.cc2013_author, ma.mover_author, ma.stayer_author, 
       prev.commits_cum, prev.owners_cum, prev.countries_cum, prev_stars.stars_cum,
       networkf.total_followers_pre2014 ,networkf.followers_ru_pre2014 ,networkf.followers_ua_pre2014 ,
       networks.total_stars_pre2014 ,networks.stars_ru_pre2014 ,networks.stars_ua_pre2014 ,
       networkc.contributors ,networkc.ua_contributors ,networkc.ru_contributors
FROM

# creates a balanced table
(SELECT a.author_id, b.country_owner, c.year, d.company_owner, e.min_year 
 FROM (SELECT DISTINCT author_id  FROM `friends-foes.commits.ru_indiv_analysis_fullcontrol`) a 
 CROSS JOIN (SELECT * FROM UNNEST(['ua_west', 'ua_east', 'ua_central', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee', 'pl', 'other']) AS country_owner) b
 CROSS JOIN (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) c 
 CROSS JOIN (SELECT * FROM UNNEST([0, 1]) AS company_owner) d
 LEFT JOIN 
     (SELECT author_id, min(year) as min_year FROM  `friends-foes.commits.ru_indiv_analysis_fullcontrol`
      GROUP BY author_id) e
ON a.author_id = e.author_id
 ) balance

# add data with commits
LEFT JOIN 
       (SELECT author_id, year, country_author, company_author, company_owner, country_partner, 
       SUM(commits_int) as commits_int, SUM(commits_ext) as commits_ext, SUM(n_owners) as n_owners, SUM(n_projects) as n_projects
       FROM `friends-foes.commits.ru_indiv_analysis_fullcontrol` 
       GROUP BY author_id, year, country_author, company_author, company_owner, country_partner
       ORDER BY author_id, year, country_partner) com 
ON com.author_id = balance.author_id and com.year = balance.year 
    and com.country_partner = balance.country_owner and com.company_owner = balance.company_owner
    
# add 2013 locations for users who were already on the platform back then
LEFT JOIN (SELECT id, min(year_created) AS year_created_a,
            min(CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #because cc2013 is not geocoded for all observations, it is replaced with the first non-missing countr
            END ) as cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            GROUP BY id)  up
ON balance.author_id = up.id


# mover or stayer
LEFT JOIN (SELECT author_id, stayer AS stayer_author, mover AS mover_author FROM `friends-foes.commits.movers`) ma
ON balance.author_id = ma.author_id

#activity controls: cumulated commits, country partners, and owners 
LEFT JOIN (SELECT year, user_id, MAX(commits_prev) as commits_cum, MAX(owners_prev) as owners_cum, 
          COUNT(DISTINCT country_partner) as countries_cum FROM `friends-foes.quality.user_commits`
          GROUP BY year, user_id) prev
ON balance.author_id = prev.user_id and balance.year = prev.year

#quality controls: number of stars 
LEFT JOIN (SELECT year, user_id, MAX(user_stars_cum) as stars_cum FROM `friends-foes.quality.users_stars`
            GROUP BY year, user_id) prev_stars
ON balance.author_id = prev_stars.user_id and balance.year = prev_stars.year 

# below added by Helene
# network heterogeneity: followers
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_followers_pre2014`) networkf
ON balance.author_id = networkf.user_id

# network heterogeneity: stars
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_stars_pre2014`) networks
ON balance.author_id = networks.user_id

# network heterogeneity: contributors
LEFT JOIN (SELECT * FROM `friends-foes.quality.users_contributors_pre2014`) networkc
ON balance.author_id = networkc.user_id
)


where year >= year_created_a

'''
client.query(query, job_config=run) 

In [ ]:
query = '''
SELECT * FROM `friends-foes.commits.ru_indiv_analysis_balanced_fullcontrol`
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regression_p2c_ru_301125-dlk.csv', encoding = 'utf-8', index=False)

#### not used anymore 

In [ ]:
# create a balanced panel: all author-ids from p2cc and all relevant countries as partner countries (drop others)
# RESULT IS LARGE, so keep RU and UA users + ua, ru, us, de, gb, ca as the main destinations 
# 'ua' 'us' 'ru' 'gb' 'de' 'fr' 'ca' 'cn' 'nl' 'in' 'br' 'au' 'pl'
query = '''
SELECT author_id, stayer_author, mover_author, cc2013_author, p.country_partner, p.year, 
        ru_identity_author,  dlk_author, year_created_a, name_class, ethnicities, confidence,
        russians, language_ru, code, district_en,
        CASE WHEN compb.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
        commits_ext, commits_int, n_owners, start_collab, quit_collab, start_project, quit_project,
        commits_total, owners_total, user_stars_prev, user_follow_prev, pr_sent_accepted/pr_sent as pr_accepted_share, 
        commits_prev_total, owners_prev_total, cp.commits_prev_country, cp.owners_prev_country
FROM
(SELECT DISTINCT author_id, stayer_author, mover_author, year_created_a, cc2013_author, b.country_partner, a.year
       FROM `friends-foes.commits.p2c` c
CROSS JOIN (SELECT * FROM UNNEST(['ua', 'us', 'ru', 'de', 'gb',  'ca', 'fr', 'cn', 'in']) AS country_partner) b
CROSS JOIN (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) a
WHERE cc2013_author in ('ua', 'ru')  #, 'us', 'ru', 'de', 'gb', 'ca'
ORDER BY author_id, year) p

# additional info from the panel
LEFT JOIN (SELECT DISTINCT id, russians, language_ru, code, district_en, ru_identity as ru_identity_author, dlk as dlk_author, year as y
            FROM `friends-foes.user_panel.ua_users_new`) ON p.author_id = id AND p.year = y
LEFT JOIN `friends-foes.user_panel.name_class` cl ON p.author_id = cl.id
LEFT JOIN `friends-foes.user_panel.name_class_web` cl2 ON p.author_id = cl2.id
LEFT JOIN `friends-foes.user_panel.sample_companies` compb ON compb.year = p.year AND compb.id = p.author_id

#activity with selected countries
LEFT JOIN (SELECT author_id as idd, year as yy, country_owner as cp,
            commits_ext, commits_int, n_owners, start_collab, quit_collab, start_project, quit_project
            FROM `friends-foes.commits.p2c`)
ON p.author_id = idd AND p.year = yy AND p.country_partner = cp

#total activity (droppping self-commits)
LEFT JOIN (SELECT DISTINCT year, author_id as user_id, 
            SUM(commits) as commits_total, COUNT(DISTINCT owner_id) as owners_total 
            FROM `friends-foes.commits.puc` 
            WHERE owner_id <> author_id
            GROUP BY author_id, year) ct
ON p.author_id = ct.user_id and p.year = ct.year

# stars  pre 2014
LEFT JOIN (SELECT user_id, MAX(user_stars_cum) As user_stars_prev
          FROM `friends-foes.quality.users_stars` WHERE year < 2014
          GROUP BY user_id) s
ON p.author_id = s.user_id 

# followers pre 2014
LEFT JOIN (SELECT user_id, MAX(user_follow_cum) As user_follow_prev
          FROM `friends-foes.quality.users_followers` WHERE year < 2014
          GROUP BY user_id) f
ON p.author_id = f.user_id 

# share of accepted pr
LEFT JOIN (SELECT user_id, MAX(pr_sent) As pr_sent, MAX(pr_sent_accepted) AS pr_sent_accepted
          FROM `friends-foes.quality.users_pull_requests` WHERE year < 2014
          GROUP BY user_id) pr
ON p.author_id = pr.user_id

# previous total commits 
LEFT JOIN (SELECT author_id as user_id,
            SUM(commits) as commits_prev_total, 
            COUNT(DISTINCT owner_id) as owners_prev_total,
          FROM `friends-foes.commits.puc` 
          WHERE year < 2014 AND owner_id <> author_id
          GROUP BY author_id) cpp
ON p.author_id = cpp.user_id 

# previous  collaborations with partner country 
LEFT JOIN (SELECT DISTINCT user_id, country_partner,
            MAX(commits_prev_country) as commits_prev_country, 
            MAX(owners_prev_country) as owners_prev_country
          FROM `friends-foes.quality.user_commits` WHERE year < 2014
          GROUP BY user_id, country_partner ) cp
ON p.author_id = cp.user_id and p.country_partner = cp.country_partner
'''

client.query(query, job_config=run) 
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_p2c_level_uaru.csv', encoding = 'utf-8', index=False)

In [ ]:
### not used anymore 


# project-level results
# for every project: number of commits to home, UA/RU, neutral camp 

query = '''
SELECT year, parent_id, owner_id, country_owner, SUM(commits), COUNT(DISTINCT author_id), 
CASE 
    WHEN country_author = country_owner THEN "home"
    WHEN country_author = 'ru' AND country_owner = 'ua' THEN "out"
    WHEN country_author = 'ua' AND country_owner = 'ru' THEN "out"
    WHEN country_author IS NULL THEN "anonymous"
    ELSE "neutral"
END region_contributor
FROM `friends-foes.commits.puc`
WHERE country_owner IN ('ru', 'ua') AND year < 2017
GROUP BY year, parent_id, owner_id, country_owner, region_contributor
'''
client.query(query, job_config=run) 
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_project_level.csv', encoding = 'utf-8', index=False)